In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'
OUTPUT_DIR = 'results'

In [4]:
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok = True)

In [5]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [6]:
metric_cols = [
    "Accuracy",
    "Weighted Accuracy",
    "Time",
    "Monotonicity",
    "Separability",
    "Linearity"
]

In [7]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    if 'dementia' not in file:

        # Process texts.
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w.lower() not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=['Total Documents', 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,0.498065,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,0.398338,4.360843,0.035360,0.267681,46.278934,17.0
4,huffPostNews,0.390427,4.199670,0.023536,0.238591,25.163032,23.0
5,medicalAbstracts,0.308262,5.109807,0.021201,0.263213,205.660064,200.0
6,simSUM,0.124588,4.106722,0.012611,0.381137,104.821400,103.0
7,syntheticCareHomeNurseNotes,0.340652,4.937623,0.027334,0.277734,26.532596,24.0
8,yahoo,0.424969,3.922505,0.029172,0.255604,47.845493,42.0


In [8]:
def make_non_normalized_dfs(input_folder, output_file_name):
    all_temp_dfs = []
    for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}'):
        if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}'):
            combination_splits = combination.split('_')
            dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}/{combination}_ksc_metrics_measures.csv')
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Dataset 1'] = dataset1
            grouped_df['Dataset 2'] = dataset2
            grouped_df['Repetitions'] = repetitions
            grouped_df['Metric'] = grouped_df.index
            grouped_df.reset_index(inplace=True)
            grouped_df.drop(columns='metric', inplace=True)
            all_temp_dfs.append(grouped_df)

    all_dfs = pd.concat(all_temp_dfs)

    print(# All datasets should have been compared the same number of times for this section to work.
    Counter(list(all_dfs['Dataset 1']) + list(all_dfs['Dataset 2'])))

    temp_dataset_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = all_dfs[
                (all_dfs['Dataset 1'] == dataset) | 
                (all_dfs['Dataset 2'] == dataset)
            ].copy()
        temp_df = temp_df.groupby('Metric')[metric_cols].mean()
        temp_df['Dataset'] = dataset
        temp_dataset_dfs.append(temp_df)
    dataset_df = pd.concat(temp_dataset_dfs)
    dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}.xlsx')

    mean_dataset_df = dataset_df.groupby('Metric')[metric_cols].mean()
    mean_dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}Mean.xlsx')

    return dataset_df, mean_dataset_df

In [9]:
ksc_dataset_df, ksc_mean_dataset_df = make_non_normalized_dfs('ksc', 'kscDataset')
ksc_synth_dataset_df, ksc_synth_mean_dataset_df = make_non_normalized_dfs('ksc_synth', 'kscSynthDataset')

Counter({'atis': 160, 'banking77': 160, 'clinc150': 160, 'clinicalDialogueSummarizations': 160, 'huffPostNews': 160, 'medicalAbstracts': 160, 'simSUM': 160, 'syntheticCareHomeNurseNotes': 160, 'yahoo': 160})
Counter({'atis': 40, 'banking77': 40, 'clinc150': 40, 'clinicalDialogueSummarizations': 40, 'huffPostNews': 40, 'medicalAbstracts': 40, 'simSUM': 40, 'syntheticCareHomeNurseNotes': 40, 'yahoo': 40})


In [10]:
ksc_dataset_df['Type'] = 'KSC'
ksc_synth_dataset_df['Type'] = 'KSC_Synth'

In [11]:
all_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Dataset', 'Metric'])[metric_cols].mean()
all_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Metric'])[metric_cols].mean()

In [12]:
all_dataset_df.to_excel(f'./{OUTPUT_DIR}/allDataset.xlsx')
all_mean_df.to_excel(f'./{OUTPUT_DIR}/allDatasetMean.xlsx')

In [13]:
all_mean_df

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity
Metric,,,,,,
CHI,0.001942,0.002920,2.228757,-0.703356,0.761050,-0.624910
CLASSIFIER,0.823742,0.763589,3.251877,0.793006,0.633384,0.815568
DC,0.848013,0.795394,0.287273,0.824229,0.815640,0.868104
FID,0.903832,0.864677,0.052551,0.814072,0.706413,0.852458
IRPR,0.736156,0.685702,24.316829,0.544084,0.320500,0.608298
MAUVE,0.897037,0.856679,0.034976,0.876096,0.827848,0.892141
PR,0.710652,0.658900,0.287324,0.559940,0.565479,0.695784
TRADITIONAL,0.894623,0.853184,0.008097,0.868727,0.779736,0.894576
ZERO,0.903242,0.866743,0.001107,0.871141,0.793407,0.895904
